In [1]:
import pandas as pd
import numpy as np
import os
import re
import json as js
from pathlib import Path
from tqdm import tqdm
import textdistance
import jiwer

In [2]:
directory_manual_entry = "Manual Entry"
directory_hasil = "Hasil Evaluasi"
os.makedirs(directory_hasil, exist_ok=True)

### Program Evaluasi

In [3]:
def find_similarity(gold_entries, entries):
    entries_result = []
    gold_entries_result = []
    similarity_values = []
    
    for i in range(len(gold_entries)):
        gold_entry = gold_entries[i]
        max_value = 0
        sentence_similar = ""

        for j in range(len(entries)):
            entry = entries[j]
            similarity = textdistance.damerau_levenshtein.normalized_similarity(gold_entry, entry)

            if (similarity > max_value):
                max_value = similarity
                sentence_similar = entry
    
        gold_entries_result.append(gold_entry)
        entries_result.append(sentence_similar)
        similarity_values.append(max_value)
    
    return gold_entries_result, entries_result, similarity_values

In [4]:
def evaluate_entry(gold_entries, entries):
    gold_entries_result, entries_result, similarity_values = find_similarity(gold_entries, entries)
    
    result = {
        "gold_entry":gold_entries_result,
        "similar_entry":entries_result,
        "similarity_values":similarity_values
    }
    
    return result

### Main Program (JSON General Font Approach)

In [5]:
def run_program_evaluation(manual_entries_directory, target_entries_directory, hasil_directory, input_type, approach_type):
    manual_entries = sorted(os.listdir(manual_entries_directory))
    entries_extraction_JSON_generic = sorted(os.listdir(target_entries_directory))
    
    kamus = []
    banyak_entri_sebenarnya = []
    banyak_entri_hasil_ekstraksi = []
    mean_similarity_tiap_entri = []
    persentasi_entri_1 = [] # diatas 0.8
    persentasi_entri_2 = [] # diatas 0.85
    persentasi_entri_3 = [] # diatas 0.9
    persentasi_entri_benar = [] # == 1
    word_error_rate = []
    char_error_rate = []
    
    for i in tqdm(range(len(manual_entries))):
        filename_manual_entry = manual_entries[i]
        filename_entry_extraction = entries_extraction_JSON_generic[i]
        print("===== Evaluasi Kamus " + filename_manual_entry + " =====")
        
        manual_entry = pd.read_csv(manual_entries_directory + "/" + filename_manual_entry)
        entry_extraction = pd.read_csv(target_entries_directory + "/" + filename_entry_extraction)
        
        entry_extraction = entry_extraction.dropna()
        entry_extraction = entry_extraction.reset_index(drop=True)
        
        entries = entry_extraction["Entri"].values.tolist()
        gold_entries = manual_entry["Entri"].values.tolist()
        
        result = evaluate_entry(gold_entries, entries)
        
        filename_result = os.path.splitext(filename_manual_entry)[0]
        filename_result = filename_result[22:]
        data_res = pd.DataFrame.from_dict(result)
        filename_res = "/Hasil Evaluasi Pendekatan " + str(approach_type) + " " + str(input_type)
        data_res.to_csv(hasil_directory + filename_res + filename_result + ".csv",index=False)
        
        kamus.append(os.path.splitext(filename_entry_extraction)[0])
        banyak_entri_sebenarnya.append(len(gold_entries))
        banyak_entri_hasil_ekstraksi.append(len(entries))

        count_true = len(data_res[data_res["similarity_values"] > 0.8])
        persen_true = round(count_true/len(gold_entries),2)
        persentasi_entri_1.append(persen_true)
        
        count_true = len(data_res[data_res["similarity_values"] > 0.85])
        persen_true = round(count_true/len(gold_entries),2)
        persentasi_entri_2.append(persen_true)
        
        count_true = len(data_res[data_res["similarity_values"] > 0.9])
        persen_true = round(count_true/len(gold_entries),2)
        persentasi_entri_3.append(persen_true)
        
        count_true = len(data_res[data_res["similarity_values"] == 1])
        persen_true = round(count_true/len(gold_entries),2)
        persentasi_entri_benar.append(persen_true)
        
        expected = ("\n").join(gold_entries)
        result_str = ("\n").join(entries)
        
        # Updated to use jiwer. Multiplied by 100 for percentage formatting.
        w_err = jiwer.wer(expected, result_str) * 100
        c_err = jiwer.cer(expected, result_str) * 100
        
        word_error_rate.append(round(w_err, 2))
        char_error_rate.append(round(c_err, 2))
        mean_similarity_tiap_entri.append(data_res["similarity_values"].mean())
        
    summarize_result = {
        "kamus" : kamus,
        "banyak_entri_sebenarnya" : banyak_entri_sebenarnya,
        "banyak_entri_pengolahan" : banyak_entri_hasil_ekstraksi,
        "mean_similarity_per_entri":mean_similarity_tiap_entri,
        "Persentase entri diatas 0.8":persentasi_entri_1,
        "Persentase entri diatas 0.85":persentasi_entri_2,
        "Persentase entri diatas 0.9":persentasi_entri_3,
        "Persentase entri benar":persentasi_entri_benar,
        "word_error_rate":word_error_rate,
        "characted_error_rate":char_error_rate
    }

    data_summarize = pd.DataFrame.from_dict(summarize_result)
    filename_summarize = "/Ringkasan Evaluasi Pendekatan " + approach_type + " bentuk " + input_type + ".csv"
    data_summarize.to_csv(hasil_directory + filename_summarize,index=False)

### JSON Font Approach

In [6]:
run_program_evaluation(
    directory_manual_entry,
    "../CSV One Entry JSON With Font Approach",
    directory_hasil,
    "JSON",
    "Font (2)"
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Manual Entry'

In [ ]:
data_summarize = pd.read_csv("Hasil Evaluasi" + "/Ringkasan Evaluasi Pendekatan Font (2) bentuk JSON.csv")
data_summarize

In [ ]:
run_program_evaluation(
    directory_manual_entry,
    "../CSV One Entry JSON With Font + Posisi Approach",
    directory_hasil,
    "JSON",
    "Font + Posisi (2)"
)

In [ ]:
data_summarize = pd.read_csv("Hasil Evaluasi" + "/Ringkasan Evaluasi Pendekatan Font + Posisi (2) bentuk JSON.csv")
data_summarize

### JSON Font + Posisi

In [ ]:
run_program_evaluation(
    directory_manual_entry,
    "CSV One Entry JSON With Font + Posisi Approach",
    "Hasil Evaluasi",
    "JSON",
    "Font_Posisi"
)

In [ ]:
data_summarize = pd.read_csv("Hasil Evaluasi" + "/Ringkasan Evaluasi Pendekatan Font_Posisi bentuk JSON.csv")
data_summarize